# Run Stage 07 with functional-hotspot and diversity upgrades

Upload your full zipped repo into `/content`, then run all cells in order.

This notebook now does five things cleanly:

1. prepares an enriched Stage 07 context with functionally weighted hotspots
2. runs multiple local ESM3 regimes across hotspot and structured-window views
3. keeps guided ESM3 attempts per sample and merges+dedoruplicates outputs
4. reranks with multimodal scoring plus regime-aware diversity selection
5. exports the top validation panel and a single downloadable Stage 07 bundle

In [ ]:
from pathlib import Path
import zipfile, shutil, os

zip_files = sorted(Path('/content').glob('*.zip'))
print('Zip files found:', [p.name for p in zip_files])
assert zip_files, 'Upload your repo zip into /content first.'
zip_path = zip_files[0]
print('Using zip:', zip_path)

work_root = Path('/content/unzipped_repo')
if work_root.exists():
    shutil.rmtree(work_root)
work_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(work_root)

repo_roots = [p for p in work_root.iterdir() if p.is_dir()]
assert repo_roots, 'Could not find extracted repo directory.'
repo_root = repo_roots[0]
os.chdir(repo_root)

print('Repo root:', repo_root)
print('Working directory:', Path.cwd())
print('Top-level files:', sorted(p.name for p in Path('.').iterdir())[:20])

In [ ]:
%pip uninstall -y hf-xet huggingface_hub -q || true
%pip install -U pip -q
%pip install -U 'pandas<3' 'huggingface_hub<1' esm transformers accelerate scikit-learn biopython -q
%pip install -e . -q
print('Install complete.')

In [ ]:
import os, getpass

os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

hf_token = getpass.getpass('Enter your Hugging Face token (Read access): ')
os.environ['HF_TOKEN'] = hf_token

print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))
print('HF_HUB_DISABLE_XET:', os.environ['HF_HUB_DISABLE_XET'])
print('PYTORCH_CUDA_ALLOC_CONF:', os.environ['PYTORCH_CUDA_ALLOC_CONF'])

In [ ]:
from huggingface_hub import hf_hub_download

probe_path = hf_hub_download(
    repo_id='EvolutionaryScale/esm3-sm-open-v1',
    filename='README.md',
    token=os.environ['HF_TOKEN'],
)
print('Downloaded probe file:', probe_path)
print('Local ESM3-open gated access looks good.')

In [ ]:
from pathlib import Path

required = [
    'scripts/07a_prepare_stage07_design_context.py',
    'scripts/07b_generate_rbps_with_esm3.py',
    'scripts/07c_score_structure_aware_candidates.py',
    'scripts/07e_rank_multimodal_candidates.py',
    'scripts/07f_make_stage07_report.py',
    'scripts/07g_export_stage07_panel.py',
    'phageforge/stage07_utils.py',
    'results/phaseA/phaseA_plan.json',
    'results/phaseA/step2_seed/phaseA_followup_seed_summary.json',
    'data/processed/rbp_dataset_eskapee_strict.csv',
]
for rel in required:
    print(rel, '->', Path(rel).exists())
assert all(Path(rel).exists() for rel in required), 'Missing required repo files.'

In [ ]:
!python scripts/07a_prepare_stage07_design_context.py \
  --phaseA_plan_json results/phaseA/phaseA_plan.json \
  --phase06c_followup_summary_json results/phaseA/step2_seed/phaseA_followup_seed_summary.json \
  --strict_csv data/processed/rbp_dataset_eskapee_strict.csv \
  --target_host Acinetobacter \
  --output_json results/stage07/context/stage07_context.base.json

In [ ]:
from pathlib import Path
import json
import pandas as pd

ctx_path = Path('results/stage07/context/stage07_context.base.json')
ctx = json.loads(ctx_path.read_text())
features = pd.DataFrame(ctx['editable_region'].get('position_features', []))
windows = pd.DataFrame(ctx['editable_region'].get('structured_windows', []))

print('selected seed id:', ctx['selected_seed']['seed_protein_id'])
print('source host:', ctx['selected_seed']['source_host'])
print('family size:', ctx['family_context']['family_member_count'])
print('family product:', ctx['family_context']['family_product_majority'])
print('num ranked hotspots:', len(ctx['editable_region'].get('hotspot_positions', [])))
print('window:', ctx['editable_region'].get('window_start'), ctx['editable_region'].get('window_end'))
print('top hotspot preview:', ctx['editable_region'].get('hotspot_positions', [])[:20])
print('\nTop functional positions:')
print(features[['position', 'seed_aa', 'consensus_aa', 'family_mutability', 'target_prior', 'functional_weight']].head(12).to_string(index=False))
print('\nStructured windows:')
print(windows[['name', 'window_start', 'window_end', 'mean_functional_weight', 'max_functional_weight']].to_string(index=False))

## Define the Stage 07 generation grid

The grid below now emphasizes:

- **two hotspot-only regimes** built from the highest functional positions
- **two structured-window regimes** built from the strongest contiguous functional windows
- **two temperature / top-k settings**
- **two seeds**

This avoids the old uniform-window failure mode.

In [ ]:
from pathlib import Path
import json
import pandas as pd

base_ctx = json.loads(Path('results/stage07/context/stage07_context.base.json').read_text())
base_hotspots = [int(x) for x in base_ctx['editable_region'].get('hotspot_positions', [])]
structured_windows = base_ctx['editable_region'].get('structured_windows', [])
assert len(structured_windows) >= 2, 'Expected at least two structured windows in the enriched context.'

generation_grid = [
    {'name': 'hotspots24_t07_k5_s42',  'editable_mode': 'hotspots_only',   'window_index': None, 'hotspot_limit': 24, 'temperature': 0.7, 'top_k': 5, 'sampling_seed': 42,  'n_samples': 8, 'esm3_num_steps': 8},
    {'name': 'hotspots32_t08_k8_s123', 'editable_mode': 'hotspots_only',   'window_index': None, 'hotspot_limit': 32, 'temperature': 0.8, 'top_k': 8, 'sampling_seed': 123, 'n_samples': 8, 'esm3_num_steps': 8},
    {'name': 'block24_t07_k5_s42',     'editable_mode': 'structured_block', 'window_index': 0,    'hotspot_limit': 24, 'temperature': 0.7, 'top_k': 5, 'sampling_seed': 42,  'n_samples': 8, 'esm3_num_steps': 8},
    {'name': 'block24_t08_k8_s123',    'editable_mode': 'structured_block', 'window_index': 1,    'hotspot_limit': 24, 'temperature': 0.8, 'top_k': 8, 'sampling_seed': 123, 'n_samples': 8, 'esm3_num_steps': 8},
]

print(pd.DataFrame(generation_grid))

In [ ]:
from pathlib import Path
import json

def write_context_variant(base_context: dict, spec: dict) -> Path:
    ctx = json.loads(json.dumps(base_context))
    edit = ctx['editable_region']
    position_features = edit.get('position_features', [])

    if spec['editable_mode'] == 'hotspots_only':
        chosen = [int(x) for x in base_hotspots[: int(spec['hotspot_limit'])]]
    elif spec['editable_mode'] == 'structured_block':
        block = structured_windows[int(spec['window_index'])]
        block_positions = [int(x) for x in block.get('positions', [])]
        chosen = block_positions[: int(spec['hotspot_limit'])]
        if not chosen:
            chosen = [int(row['position']) for row in position_features[: int(spec['hotspot_limit'])]]
    else:
        raise ValueError(f"Unknown editable_mode: {spec['editable_mode']}")

    chosen = sorted(set(chosen))
    feature_lookup = {int(row['position']): row for row in position_features}
    edit['hotspot_positions'] = chosen
    edit['hotspot_priority_weights'] = {str(pos): float(feature_lookup.get(pos, {}).get('functional_weight', 0.0)) for pos in chosen}
    edit['position_features'] = [feature_lookup[pos] for pos in chosen if pos in feature_lookup]
    edit['window_start'] = int(min(chosen))
    edit['window_end'] = int(max(chosen) + 1)

    out_path = Path('results/stage07/context') / f"stage07_context.{spec['name']}.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(ctx, indent=2))
    return out_path

for spec in generation_grid:
    p = write_context_variant(base_ctx, spec)
    print('Wrote context:', p)

In [ ]:
from pathlib import Path
import subprocess, shlex, pandas as pd

gen_root = Path('results/stage07/generation')
gen_root.mkdir(parents=True, exist_ok=True)

for rel in gen_root.glob('*.csv'):
    rel.unlink()

per_regime_paths = []
for spec in generation_grid:
    context_json = Path('results/stage07/context') / f"stage07_context.{spec['name']}.json"
    out_csv = gen_root / f"generated_{spec['name']}.csv"

    cmd = [
        'python', 'scripts/07b_generate_rbps_with_esm3.py',
        '--context_json', str(context_json),
        '--out_csv', str(out_csv),
        '--n_samples', str(spec['n_samples']),
        '--temperature', str(spec['temperature']),
        '--top_k', str(spec['top_k']),
        '--esm3_backend', 'local',
        '--esm3_model', 'esm3-open',
        '--sampling_seed', str(spec['sampling_seed']),
        '--max_esm3_masked_positions', str(spec['hotspot_limit']),
        '--esm3_num_steps', str(spec['esm3_num_steps']),
        '--max_attempts_per_sample', '3',
        '--esm3_error_fallback', 'none',
    ]
    print('\nRUNNING:', ' '.join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd, check=True)

    df = pd.read_csv(out_csv)
    df['generation_regime'] = spec['name']
    df['editable_mode'] = spec['editable_mode']
    df['regime_hotspot_limit'] = spec['hotspot_limit']
    df['regime_temperature'] = spec['temperature']
    df['regime_top_k'] = spec['top_k']
    df['regime_sampling_seed'] = spec['sampling_seed']
    df.to_csv(out_csv, index=False)
    per_regime_paths.append(str(out_csv))

print('\nPer-regime generation CSVs:')
for p in per_regime_paths:
    print(p)

In [ ]:
from pathlib import Path
import pandas as pd

all_frames = [pd.read_csv(Path(p)) for p in per_regime_paths]
all_generated = pd.concat(all_frames, ignore_index=True)
all_generated['candidate_sequence'] = all_generated['candidate_sequence'].astype(str)

dedup_generated = (
    all_generated
    .sort_values(
        ['generation_status', 'candidate_sequence', 'guided_mutation_score', 'mutation_penalty', 'regime_hotspot_limit'],
        ascending=[True, True, False, True, True],
        kind='mergesort',
    )
    .drop_duplicates(subset=['candidate_sequence'], keep='first')
    .reset_index(drop=True)
)
dedup_generated['sample_id'] = range(len(dedup_generated))

merged_csv = Path('results/stage07/generation/all_generated_candidates.csv')
regime_summary_csv = Path('results/stage07/generation/generation_regime_summary.csv')

dedup_generated.to_csv(merged_csv, index=False)

regime_summary = (
    all_generated
    .groupby('generation_regime', dropna=False)
    .agg(
        rows_generated=('candidate_sequence', 'size'),
        unique_sequences=('candidate_sequence', 'nunique'),
        ok_rows=('generation_status', lambda s: int((s.astype(str) == 'ok').sum())),
        mean_mutation_penalty=('mutation_penalty', 'mean'),
        mean_guided_score=('guided_mutation_score', 'mean'),
    )
    .reset_index()
    .sort_values('generation_regime')
)
regime_summary.to_csv(regime_summary_csv, index=False)

print('Merged rows before dedup:', len(all_generated))
print('Merged rows after dedup:', len(dedup_generated))
print('Wrote merged CSV:', merged_csv)
print('Wrote regime summary CSV:', regime_summary_csv)
print(regime_summary.to_string(index=False))

In [ ]:
gen = pd.read_csv('results/stage07/generation/all_generated_candidates.csv')
print(gen.head(10).to_string())
print('\ngenerator modes:', gen['generator_mode'].value_counts(dropna=False).to_dict())
print('generation status:', gen['generation_status'].value_counts(dropna=False).to_dict())
print('generation regimes:', gen['generation_regime'].value_counts(dropna=False).to_dict())
print('unique candidate sequences:', gen['candidate_sequence'].nunique())

In [ ]:
!python scripts/07c_score_structure_aware_candidates.py \
  --context_json results/stage07/context/stage07_context.base.json \
  --generated_csv results/stage07/generation/all_generated_candidates.csv \
  --scored_csv results/stage07/structure_rerank/structure_scored_candidates.csv

In [ ]:
scored = pd.read_csv('results/stage07/structure_rerank/structure_scored_candidates.csv')
print(scored.head(10).to_string())
print(scored.columns.tolist())

In [ ]:
!python scripts/07e_rank_multimodal_candidates.py \
  --generated_csv results/stage07/generation/all_generated_candidates.csv \
  --structure_scored_csv results/stage07/structure_rerank/structure_scored_candidates.csv \
  --diverse_top_k 5 \
  --per_regime_pool 3 \
  --out_csv results/stage07/multimodal_rank/final_multimodal_ranked_candidates.csv

In [ ]:
ranked = pd.read_csv('results/stage07/multimodal_rank/final_multimodal_ranked_candidates.csv')
print(ranked.head(10).to_string())
print(ranked.columns.tolist())

In [ ]:
!python scripts/07f_make_stage07_report.py \
  --context_json results/stage07/context/stage07_context.base.json \
  --ranked_csv results/stage07/multimodal_rank/final_multimodal_ranked_candidates.csv \
  --report_md results/stage07/report/stage07_report.md

!python scripts/07g_export_stage07_panel.py \
  --ranked_csv results/stage07/multimodal_rank/final_multimodal_ranked_candidates.csv \
  --output_dir results/stage07/exports \
  --top_k 5

In [ ]:
from pathlib import Path

report_path = Path('results/stage07/report/stage07_report.md')
panel_csv = Path('results/stage07/exports/top5_validation_panel.csv')
panel_fasta = Path('results/stage07/exports/top5_validation_panel.fasta')

print('report exists:', report_path.exists(), report_path)
print('panel csv exists:', panel_csv.exists(), panel_csv)
print('panel fasta exists:', panel_fasta.exists(), panel_fasta)

if report_path.exists():
    print('\n--- report preview ---\n')
    print(report_path.read_text()[:3000])

if panel_fasta.exists():
    print('\n--- FASTA preview ---\n')
    print(panel_fasta.read_text()[:2000])

In [ ]:
top5 = ranked.sort_values(['rank_diverse', 'rank_raw']).head(5).copy()
print(top5[['sample_id', 'generation_regime', 'final_multimodal_rank_score', 'target_score', 'strict_manifold_score', 'structure_score', 'used_local_esm3', 'used_esm3_api', 'used_esm2_fallback', 'mutation_positions']].to_string(index=False))
print('\nTop candidate sample IDs:', top5['sample_id'].tolist())
print('Top candidate generation regimes:', top5['generation_regime'].tolist())
print('Top scores:', [round(float(x), 6) for x in top5['final_multimodal_rank_score']])
print('Local ESM3 used:', bool(ranked['used_local_esm3'].any()))
print('ESM3 API used:', bool(ranked['used_esm3_api'].any()))
print('ESM2 fallback used:', bool(ranked['used_esm2_fallback'].any()))

In [ ]:
from pathlib import Path
import zipfile

bundle_files = [
    'results/stage07/context/stage07_context.base.json',
    'results/stage07/context/stage07_context.hotspots24_t07_k5_s42.json',
    'results/stage07/context/stage07_context.hotspots32_t08_k8_s123.json',
    'results/stage07/context/stage07_context.block24_t07_k5_s42.json',
    'results/stage07/context/stage07_context.block24_t08_k8_s123.json',
    'results/stage07/generation/all_generated_candidates.csv',
    'results/stage07/generation/generation_regime_summary.csv',
    'results/stage07/structure_rerank/structure_scored_candidates.csv',
    'results/stage07/multimodal_rank/final_multimodal_ranked_candidates.csv',
    'results/stage07/multimodal_rank/top_validation_panel.csv',
    'results/stage07/report/stage07_report.md',
    'results/stage07/exports/top5_validation_panel.csv',
    'results/stage07/exports/top5_validation_panel.fasta',
    'results/stage07/exports/top5_validation_panel.json',
]

for p in Path('results/stage07/generation').glob('generated_*.csv'):
    bundle_files.append(str(p))

zip_path = Path('results/stage07_bundle.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for rel in bundle_files:
        path = Path(rel)
        if path.exists():
            zf.write(path, arcname=rel)

print('Created zip:', zip_path)
for rel in sorted(bundle_files):
    if Path(rel).exists():
        print(rel)